In [ ]:
import matplotlib.pyplot as plt

# 设置支持中文的字体（例如 SimHei），同时确保负号能正常显示
plt.rcParams['font.sans-serif'] = ['Times New Roman', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 可视化绘图

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from matplotlib_scalebar.scalebar import ScaleBar
from shapely.geometry import Point

# ----------------配置区域----------------
# 1. 定义文件路径
csv_path = '../data/7.匹配时间/merged_output.csv'

# 2. 定义城市中心点坐标 (经度 lng, 纬度 lat) - WGS84坐标系
# 这些是大致的市中心坐标
city_centers = {
    'Beijing': (116.4074, 39.9042),
    'Shanghai': (121.4737, 31.2304),
    'Guangzhou': (113.2644, 23.1291),
    'Shenzhen': (114.0579, 22.5431),
    'Chengdu': (104.0668, 30.5728),
    'Tianjin': (117.2009, 39.0842)
}

# 3. 定义可视化参数
radius_m = 30000  # 半径20km，即向外扩20000米
plot_crs = "EPSG:3857"  # Web Mercator投影，用于底图和距离计算

# ----------------数据处理----------------
print("正在读取数据...")
# 读取CSV
df = pd.read_csv(csv_path)

# 转换为GeoDataFrame，原始坐标默认为 EPSG:4326 (经纬度)
gdf = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df.lng, df.lat), 
    crs="EPSG:4326"
)

# 转换投影到 Web Mercator (米为单位)，以便计算距离和叠加底图
print("正在进行坐标投影转换...")
gdf_3857 = gdf.to_crs(plot_crs)

# ----------------绘图逻辑----------------
print("开始绘图...")

# 创建 2行3列 的子图布局
fig, axes = plt.subplots(2, 3, figsize=(18, 12), constrained_layout=True)
axes_flat = axes.flatten()  # 展平以便遍历

for i, (city_name, (lng, lat)) in enumerate(city_centers.items()):
    ax = axes_flat[i]
    
    # 1. 计算该城市的中心点和范围 (在 EPSG:3857 下)
    # 先将中心点转为 Point 对象并转投影
    center_pt = gpd.GeoSeries([Point(lng, lat)], crs="EPSG:4326").to_crs(plot_crs)
    center_x = center_pt[0].x
    center_y = center_pt[0].y
    
    # 计算外接矩形范围 (中心点 +/- 半径)
    min_x, max_x = center_x - radius_m, center_x + radius_m
    min_y, max_y = center_y - radius_m, center_y + radius_m
    
    # 2. 空间索引/裁剪数据 (为了提高绘图速度，只绘制范围内的点)
    # 使用 cx 进行空间切片，只获取该城市矩形框内的数据
    city_data = gdf_3857.cx[min_x:max_x, min_y:max_y]
    
    # 3. 绘制散点
    # s=5 为点的大小，根据数据量可适当调整
    city_data.plot(ax=ax, color='#7098d8', markersize=4, alpha=0.7)
    
    # 4. 设置地图范围 (确保是正方形)
    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.set_aspect('equal') # 强制正方形比例
    
    # 5. 添加底图 (CartoDB Positron)
    # zoom参数控制底图清晰度，自动计算或手动指定(如 zoom=12)
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, crs=plot_crs)
    
    # 6. 左上角添加城市名称
    # transform=ax.transAxes 使得坐标基于子图相对位置 (0,1) 为左上角
    ax.text(0.05, 0.95, city_name, transform=ax.transAxes, 
            fontsize=24, fontweight='bold', va='top', ha='left',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    
    # 7. 右下角添加2km比例尺
    # fixed_value=2000 强制显示2km (需要matplotlib-scalebar较新版本)
    # 如果报错，可以去掉 fixed_value，它会自动计算合适的比例
    scalebar = ScaleBar(
        dx=1,  # 1个单位 = 1米 (EPSG:3857)
        units="m", 
        location="lower right", 
        fixed_value=4000, # 固定显示2km
        scale_loc="bottom", 
        color='black',
        box_alpha=0.5 # 背景透明度
    )
    ax.add_artist(scalebar)
    
    # 移除坐标轴刻度让图更清爽
    ax.set_xticks([])
    ax.set_yticks([])

# 保存或显示
# plt.suptitle("Transport Visualizations by City (20km Radius)", fontsize=20)

# plt.savefig('city_visualization.png', dpi=300) # 如果需要保存取消注释
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from matplotlib_scalebar.scalebar import ScaleBar
from shapely.geometry import Point

# ----------------配置区域----------------
# 1. 定义文件路径
csv_path = '../data/7.匹配时间/merged_output.csv'

# 2. 定义城市中心点坐标 (经度 lng, 纬度 lat) - WGS84坐标系
# 这些是大致的市中心坐标
city_centers = {
    'Beijing': (116.4074, 39.9042),
    'Shanghai': (121.4737, 31.2304),
    'Guangzhou': (113.2644, 23.1291),
    'Shenzhen': (114.0579, 22.5431),
    'Chengdu': (104.0668, 30.5728),
    'Tianjin': (117.2009, 39.0842)
}

# 3. 定义可视化参数
radius_m = 30000  # 半径20km，即向外扩20000米
plot_crs = "EPSG:3857"  # Web Mercator投影，用于底图和距离计算

# ----------------数据处理----------------
print("正在读取数据...")
# 读取CSV
df = pd.read_csv(csv_path)

# 转换为GeoDataFrame，原始坐标默认为 EPSG:4326 (经纬度)
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.lng, df.lat),
    crs="EPSG:4326"
)

# 转换投影到 Web Mercator (米为单位)，以便计算距离和叠加底图
print("正在进行坐标投影转换...")
gdf_3857 = gdf.to_crs(plot_crs)

# ----------------绘图逻辑----------------
print("开始绘图...")

# 创建 2行3列 的子图布局
fig, axes = plt.subplots(2, 3, figsize=(18, 12), constrained_layout=True)
axes_flat = axes.flatten()  # 展平以便遍历

for i, (city_name, (lng, lat)) in enumerate(city_centers.items()):
    ax = axes_flat[i]

    # 1. 计算该城市的中心点和范围 (在 EPSG:3857 下)
    # 先将中心点转为 Point 对象并转投影
    center_pt = gpd.GeoSeries([Point(lng, lat)], crs="EPSG:4326").to_crs(plot_crs)
    center_x = center_pt[0].x
    center_y = center_pt[0].y

    # 计算外接矩形范围 (中心点 +/- 半径)
    min_x, max_x = center_x - radius_m, center_x + radius_m
    min_y, max_y = center_y - radius_m, center_y + radius_m

    # 2. 空间索引/裁剪数据 (为了提高绘图速度，只绘制范围内的点)
    # 使用 cx 进行空间切片，只获取该城市矩形框内的数据
    city_data = gdf_3857.cx[min_x:max_x, min_y:max_y]

    # 3. 绘制散点
    # s=5 为点的大小，根据数据量可适当调整
    city_data.plot(ax=ax, color='#7098d8', markersize=4, alpha=0.7)

    # 4. 设置地图范围 (确保是正方形)
    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.set_aspect('equal')  # 强制正方形比例

    # 5. 添加底图 (CartoDB Positron)
    # zoom参数控制底图清晰度，自动计算或手动指定(如 zoom=12)
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, crs=plot_crs)

    # 6. 左上角添加城市名称
    # transform=ax.transAxes 使得坐标基于子图相对位置 (0,1) 为左上角
    ax.text(0.05, 0.95, city_name, transform=ax.transAxes,
            fontsize=24, fontweight='bold', va='top', ha='left',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    # 7. 右下角添加2km比例尺
    # fixed_value=2000 强制显示2km (需要matplotlib-scalebar较新版本)
    # 如果报错，可以去掉 fixed_value，它会自动计算合适的比例
    #
    # >>> 修改点 1：把显示从 4000m 改为 4km（显示单位改为 km）
    # >>> 修改点 2：比例尺整体变大（更长、更粗、字体更大）
    #
    # 说明：
    # - EPSG:3857 的坐标单位是“米”，所以 dx=1 表示 1个数据单位=1米
    # - 优先使用 fixed_units="km"（新版本 matplotlib-scalebar 支持），这样既保持 dx=1(米)，又能显示 km
    # - 若你的版本不支持 fixed_units，则回退方案：把 units="km"，同时 dx 改为 0.001（因为 1米=0.001km）
    try:
        scalebar = ScaleBar(
            dx=1,  # 1个单位 = 1米 (EPSG:3857)
            units="m",
            location="lower right",
            fixed_value=10,          # 固定显示 4
            fixed_units="km",       # 固定单位为 km -> 显示为 4 km
            scale_loc="bottom",
            color='black',
            box_alpha=0.5,          # 背景透明度

            # 让比例尺整体变大（不同版本参数名可能略有差异，但这些在常见版本里可用）
            length_fraction=0.30,   # 比例尺长度占子图宽度比例（默认更短，调大更显眼）
            width_fraction=0.02,    # 比例尺厚度（调大更粗）
            font_properties={"size": 14}  # 字体变大
        )
    except TypeError:
        # 兼容旧版本：不支持 fixed_units 时，用 km 作为显示单位，并把 dx 转为 km/数据单位
        scalebar = ScaleBar(
            dx=0.001,  # 1个单位(米) = 0.001 km
            units="km",
            location="lower right",
            fixed_value=10,          # 固定显示 4 km
            scale_loc="bottom",
            color='black',
            box_alpha=0.5,

            length_fraction=0.30,
            width_fraction=0.03,
            font_properties={"size": 14}
        )

    ax.add_artist(scalebar)

    # 移除坐标轴刻度让图更清爽
    ax.set_xticks([])
    ax.set_yticks([])

# 保存或显示
# plt.suptitle("Transport Visualizations by City (20km Radius)", fontsize=20)

plt.savefig('../data/figure/city_visualization.png', dpi=300) # 如果需要保存取消注释
plt.show()
